In [2]:
import os
import re

def load_dataset(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file: {path}")

    valid_students = []

    with open(path, 'r', encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            try:
                parts = line.split('|')

                if len(parts) < 7:
                    raise ValueError("Thiếu dữ liệu")

                msv, hoten, namsinh, nganh, toan, ly, hoa = parts[:7]

                msv = msv.strip()
                hoten = hoten.strip()
                namsinh = namsinh.strip()
                nganh = nganh.strip()

                if not (msv and hoten and namsinh and nganh):
                    raise ValueError("Thiếu thông tin")

                try:
                    namsinh = int(namsinh)
                except:
                    raise ValueError("Năm sinh không hợp lệ")

                try:
                    toan = float(toan)
                    ly = float(ly)
                    hoa = float(hoa)
                except:
                    raise ValueError("Điểm không phải số")

                for mon, diem in zip(["Toán", "Lý", "Hóa"], [toan, ly, hoa]):
                    if diem < 0 or diem > 10:
                        raise ValueError(f"{mon} ngoài khoảng 0-10")

                email = ""
                sdt = ""

                if len(parts) >= 9:
                    email = parts[7].strip()
                    sdt = parts[8].strip()

                    if not email_hople(email):
                        raise ValueError("Email không hợp lệ")

                    if not sdt_hople(sdt):
                        raise ValueError("SĐT không hợp lệ")

                valid_students.append(
                    [msv, hoten, namsinh, nganh, toan, ly, hoa, email, sdt]
                )

            except ValueError as e:
                print(f"Dòng {line_number} lỗi: {line} -> {e}")
            except Exception:
                print(f"Dòng {line_number} sai định dạng: {line}")

    return valid_students

def email_hople(email):
    if not email:
        return True
    pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    return re.match(pattern, email)


def sdt_hople(sdt):
    if not sdt:
        return True
    return sdt.isdigit() and (9 <= len(sdt) <= 11)

def diemtrungbinh(toan, ly, hoa):
    return (toan + ly + hoa) / 3


def xeploai(avg):
    if avg >= 8.5:
        return "Giỏi"
    elif avg >= 7:
        return "Khá"
    elif avg >= 5:
        return "Trung bình"
    else:
        return "Yếu"

def hien_thi(ds):
    for sv in ds:
        msv, hoten, namsinh, nganh, toan, ly, hoa = sv[:7]
        avg = diemtrungbinh(toan, ly, hoa)
        print(f"{msv} - {hoten} - {nganh} - TB: {avg:.2f} - {xeploai(avg)}")

def loc_sinhvien_dat(ds):
    return [sv for sv in ds if diemtrungbinh(sv[4], sv[5], sv[6]) >= 7]


def sinhvien_max(ds):
    return max(ds, key=lambda sv: diemtrungbinh(sv[4], sv[5], sv[6]))


def thongke_nganh(ds):
    result = {}
    for sv in ds:
        nganh = sv[3]
        result[nganh] = result.get(nganh, 0) + 1
    return result


def sapxep(ds):
    return sorted(ds, key=lambda sv: diemtrungbinh(sv[4], sv[5], sv[6]), reverse=True)

def xuat(ds, filename="ketqua.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for sv in ds:
            msv, hoten, _, nganh, toan, ly, hoa = sv[:7]
            avg = diemtrungbinh(toan, ly, hoa)
            f.write(f"{msv} - {hoten} - {nganh} - TB: {avg:.2f} - {xeploai(avg)}\n")


def ghi_passed(ds):
    ds_dat = loc_sinhvien_dat(ds)
    xuat(ds_dat, "passed.txt")

def nhap_sinhvien():
    try:
        msv = input("MSV: ")
        hoten = input("Họ tên: ")
        namsinh = int(input("Năm sinh: "))
        nganh = input("Ngành: ")

        toan = float(input("Toán: "))
        ly = float(input("Lý: "))
        hoa = float(input("Hóa: "))

        email = input("Email (có thể bỏ trống): ")
        sdt = input("SĐT (có thể bỏ trống): ")

        if not email_hople(email):
            raise ValueError("Email sai")

        if not sdt_hople(sdt):
            raise ValueError("SĐT sai")

        for d in [toan, ly, hoa]:
            if d < 0 or d > 10:
                raise ValueError("Điểm không hợp lệ")

        return [msv, hoten, namsinh, nganh, toan, ly, hoa, email, sdt]

    except Exception as e:
        print("Lỗi:", e)
        return None

def tim_sinhvien(ds):
    key = input("Nhập MSV hoặc tên: ").lower()

    kq = [sv for sv in ds if key in sv[0].lower() or key in sv[1].lower()]

    if kq:
        hien_thi(kq)
    else:
        print("Không tìm thấy")

def menu():
    path = r"D:\Python baitap\BTH\Python\BTH6\student.txt"

    try:
        ds = load_dataset(path)
    except Exception as e:
        print(e)
        return

    while True:
        print("\n===== MENU =====")
        print("1. Xem danh sách")
        print("2. Tìm sinh viên")
        print("3. Thống kê")
        print("4. Thoát")

        chon = input("Chọn: ")

        if chon == "1":
            hien_thi(ds)

        elif chon == "2":
            tim_sinhvien(ds)

        elif chon == "3":
            tk = thongke_nganh(ds)
            for nganh, sl in tk.items():
                print(f"{nganh}: {sl}")

        elif chon == "4":
            print("Thoát.")
            break

        else:
            print("Chọn sai!")

def main():
    menu()
if __name__ == "__main__":
    main()

Dòng 2 lỗi: SV002|Tran Thi B|2003|KinhTe|6.0|abc|7.5 -> Điểm không phải số
Dòng 4 lỗi: SV004|Pham Thi D|2002|YDuoc|5.5|6.0|-1 -> Hóa ngoài khoảng 0-10
Dòng 5 lỗi: SV005|Hoang Van E|abcd|CNTT|7.0|8.0|9.0 -> Năm sinh không hợp lệ

===== MENU =====
1. Xem danh sách
2. Tìm sinh viên
3. Thống kê
4. Thoát
SV001 - Nguyen Van A - CNTT - TB: 8.17 - Khá
SV003 - Le Van C - CNTT - TB: 9.17 - Giỏi
SV006 - Do Thi F - CNTT - TB: 9.33 - Giỏi
SV007 - Nguyen Nhu - YDuoc - TB: 6.17 - Trung bình

===== MENU =====
1. Xem danh sách
2. Tìm sinh viên
3. Thống kê
4. Thoát
SV001 - Nguyen Van A - CNTT - TB: 8.17 - Khá
SV003 - Le Van C - CNTT - TB: 9.17 - Giỏi
SV006 - Do Thi F - CNTT - TB: 9.33 - Giỏi
SV007 - Nguyen Nhu - YDuoc - TB: 6.17 - Trung bình

===== MENU =====
1. Xem danh sách
2. Tìm sinh viên
3. Thống kê
4. Thoát
Không tìm thấy

===== MENU =====
1. Xem danh sách
2. Tìm sinh viên
3. Thống kê
4. Thoát
Chọn sai!

===== MENU =====
1. Xem danh sách
2. Tìm sinh viên
3. Thống kê
4. Thoát
Chọn sai!

===== MENU 

KeyboardInterrupt: Interrupted by user